In [1]:
import os

os.chdir("/nfs/home11/ugrad/2024/es348865/atm306/Final_Project")
print(os.getcwd())

/nfs/home11/ugrad/2024/es348865/atm306/Final_Project


In [2]:
with open("data_MJO_CPC_phase.txt", "r", encoding="latin-1") as f:
    print(f.readline())

           INDEX_9  INDEX_10  INDEX_1  INDEX_2  INDEX_3  INDEX_4  INDEX_5  INDEX_6  INDEX_7  INDEX_8



In [3]:
import pandas as pd

df = pd.read_csv(
    "data_MJO_CPC_phase.txt",
    delim_whitespace=True,
    skiprows=1  # skips the weird header formatting line
)

df.head()

/tmp/ipykernel_2483367/2257838694.py:3: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(


,PENTAD,20E,70E,80E,100E,120E,140E,160E,120W,40W,10W
0,19780103,0.38,1.01,1.28,0.73,0.54,-0.16,-0.77,-1.22,-1.09,-0.34
1,19780108,-0.99,-0.27,0.82,1.39,1.34,1.10,0.57,-0.26,-1.18,-1.36
2,19780113,-1.10,-1.17,-0.65,0.33,0.60,1.03,1.18,0.96,0.14,-0.69
3,19780118,1.68,0.88,-0.67,-1.78,-1.70,-1.73,-1.25,-0.20,1.25,1.88
4,19780123,2.29,2.45,1.33,-0.69,-1.12,-2.12,-2.50,-2.07,-0.37,1.40


In [5]:
import pandas as pd
import numpy as np

# ----------------------------
# 1. Load data
# ----------------------------
df = pd.read_csv(
    "data_MJO_CPC_phase.txt",
    sep=r"\s+",
    skiprows=1
)

# ----------------------------
# 2. Rename time column
# ----------------------------
df = df.rename(columns={"PENTAD": "date"})

# ----------------------------
# 3. Convert missing-value symbols to NaN
# ----------------------------
df = df.replace(["*****", "****", "***", "**", "*"], np.nan)

# ----------------------------
# 4. Convert date column
# ----------------------------
df["date"] = pd.to_datetime(df["date"], format="%Y%m%d")

# ----------------------------
# 5. Convert all MJO columns to numeric safely
# ----------------------------
cols = df.columns.drop("date")
df[cols] = df[cols].apply(pd.to_numeric, errors="coerce")

# ----------------------------
# 6. Build CPC-style MJO index
# ----------------------------

# Option A: global mean state
df["mjo_index"] = df.loc[:, "20E":"10W"].mean(axis=1)

# OPTIONAL (better wave structure):
# df["mjo_index"] = df["120W"] - df["100E"]

# ----------------------------
# 7. Define regimes
# ----------------------------
df["regime"] = "neutral"

df.loc[df["mjo_index"] > 0.5, "regime"] = "positive"
df.loc[df["mjo_index"] < -0.5, "regime"] = "negative"

# ----------------------------
# 8. Sanity checks
# ----------------------------
print(df.head())
print(df["regime"].value_counts())

        date   20E   70E   80E  100E  120E  140E  160E  120W   40W   10W  \
0 1978-01-03  0.38  1.01  1.28  0.73  0.54 -0.16 -0.77 -1.22 -1.09 -0.34   
1 1978-01-08 -0.99 -0.27  0.82  1.39  1.34  1.10  0.57 -0.26 -1.18 -1.36   
2 1978-01-13 -1.10 -1.17 -0.65  0.33  0.60  1.03  1.18  0.96  0.14 -0.69   
3 1978-01-18  1.68  0.88 -0.67 -1.78 -1.70 -1.73 -1.25 -0.20  1.25  1.88   
4 1978-01-23  2.29  2.45  1.33 -0.69 -1.12 -2.12 -2.50 -2.07 -0.37  1.40   

   mjo_index   regime  
0      0.036  neutral  
1      0.116  neutral  
2      0.063  neutral  
3     -0.164  neutral  
4     -0.140  neutral  
regime
neutral    3577
Name: count, dtype: int64


In [6]:
import pandas as pd
import numpy as np
from scipy.stats import binomtest

# ============================================================
# 1. LOAD CPC VELOCITY POTENTIAL DATA
# ============================================================
df = pd.read_csv(
    "data_MJO_CPC_phase.txt",
    sep=r"\s+",
    skiprows=1
)

# Rename time column
df = df.rename(columns={"PENTAD": "date"})

# Clean missing values
df = df.replace(["*****", "****", "***", "**", "*"], np.nan)

# Convert types
df["date"] = pd.to_datetime(df["date"], format="%Y%m%d")
cols = df.columns.drop("date")
df[cols] = df[cols].apply(pd.to_numeric, errors="coerce")

# ============================================================
# 2. BUILD CPC-STYLE MJO INDEX (PAPER-BASED DIPOLE)
#    120W - 100E captures Pacific–Atlantic convective contrast
# ============================================================
df["mjo_index"] = df["120W"] - df["100E"]

# Optional normalization (recommended for robustness)
df["mjo_index"] = (
    df["mjo_index"] / df.loc[:, "20E":"10W"].std(axis=1)
)

# ============================================================
# 3. DEFINE CPC 3 REGIMES
# ============================================================
df["phase3"] = "neutral"

df.loc[df["mjo_index"] > 0.5, "phase3"] = "positive"
df.loc[df["mjo_index"] < -0.5, "phase3"] = "negative"

# ============================================================
# 4. CHECK DISTRIBUTION OF REGIMES
# ============================================================
print("\nCPC Regime Counts:")
print(df["phase3"].value_counts())

# ============================================================
# 5. MERGE WITH TC GENESIS DATA
#    (assumes you already have a dataframe called genesis)
# ============================================================
# Example structure assumption:
# genesis columns: ['storm', 'date', 'lat', 'lon', ...]

genesis = genesis.copy()
genesis["date"] = pd.to_datetime(genesis["date"])

genesis = genesis.merge(
    df[["date", "phase3"]],
    on="date",
    how="left"
)

# Remove unmatched dates
genesis = genesis.dropna(subset=["phase3"])

# ============================================================
# 6. BASIN SPLIT (ATLANTIC vs EASTERN/CENTRAL PACIFIC)
# ============================================================
atlantic = genesis[genesis["lon"] > -80]
pacific  = genesis[genesis["lon"] <= -80]

# ============================================================
# 7. FREQUENCY OF GENESIS BY MJO PHASE
# ============================================================
atl_freq = atlantic["phase3"].value_counts(normalize=True)
pac_freq = pacific["phase3"].value_counts(normalize=True)

print("\nAtlantic Genesis Frequency by Phase:")
print(atl_freq)

print("\nPacific Genesis Frequency by Phase:")
print(pac_freq)

# ============================================================
# 8. BINOMIAL SIGNIFICANCE TESTS
#    (Paper-style significance of modulation)
# ============================================================

n_total = len(genesis)
p_expected = 1/3  # equal probability null hypothesis

results = {}

for basin_name, data in [("Atlantic", atlantic), ("Pacific", pacific)]:
    for phase in ["positive", "negative"]:
        x = (data["phase3"] == phase).sum()
        n = len(data)

        test = binomtest(x, n=n, p=p_expected)

        results[(basin_name, phase)] = {
            "count": x,
            "n": n,
            "fraction": x/n,
            "p_value": test.pvalue
        }

# Print results
print("\nBinomial Test Results:")
for k, v in results.items():
    print(f"{k}: {v}")

# ============================================================
# 9. DIPLOLE SUMMARY (PAPER CORE RESULT STYLE)
# ============================================================

summary = pd.DataFrame(results).T
print("\nSummary Table:")
print(summary)


CPC Regime Counts:
phase3
positive    1572
negative    1406
neutral      599
Name: count, dtype: int64


NameError: name 'genesis' is not defined